# Piloto — Batch-balanced Focal Loss (BBFL) en RoBERTuito

**Pregunta.** ¿La *Batch-balanced Focal Loss* de Singh et al. (2023) —Focal Loss con lotes de entrenamiento balanceados por clase— mejora a RoBERTuito frente a la configuración actual?

Es un **piloto**: usa **solo el train oficial** y **no toca el test**. Reutiliza sin cambios la carga de datos, los grupos (`GROUP_ID`), el tokenizador, el modelo, el LR por capas y los argumentos de entrenamiento de `04c_RoBERTuito.ipynb`, y la **configuración final** de cada variante (`robertuito_config_final.json`). Solo cambian dos cosas: la función de error y cómo se arman los lotes.

## Diseño (fijado antes de ejecutar)

| Opción | Función de error | Pesos de clase | Label smoothing | Lotes de entrenamiento |
|---|---|---|---|---|
| `REF` | Entropía cruzada (configuración actual) | balanced | 0.1 | habituales (aleatorios) |
| `CE` | Entropía cruzada | no | 0 | habituales |
| `FL` | Focal Loss, γ = 2 | no | 0 | habituales |
| `CE_bal` | Entropía cruzada | no | 0 | **balanceados: 8 irónicos + 8 no irónicos** |
| `BBFL` | Focal Loss, γ = 2 | no | 0 | **balanceados: 8 + 8** |

Las cuatro opciones nuevas forman un diseño **2 × 2** (función de error × tipo de lote), así que se puede separar qué aporta el balanceo y qué aporta Focal Loss.

**Lo que se mantiene igual en las 5 opciones:**

- los mismos **3 folds agrupados** del train (`StratifiedGroupKFold`, `random_state=SEED`), los mismos de la optimización final;
- la misma **semilla por fold**, y cada entrenamiento parte del **checkpoint preentrenado** (`crear_modelo`), nunca del modelo final ajustado;
- los mismos hiperparámetros finales de la variante (LR 2e-5, batch 16, weight decay, dropout, LR por capas 0.9, máscara de tokens);
- el mismo **presupuesto**: `num_train_epochs` épocas con el mismo número de pasos por época, ⌈n_train / 16⌉, y el mismo *scheduler*. Los lotes balanceados no añaden pasos: en cada época los irónicos (≈ 1/3 del train) se repiten más y los no irónicos se ven algo menos;
- la **validación conserva su distribución original** (≈ 1/3 irónicos) y no usa pesos ni label smoothing.

**Lotes balanceados.** Cada lote tiene exactamente 8 irónicos y 8 no irónicos. Se recorre cada clase en orden aleatorio sin repetir hasta agotarla y luego se vuelve a barajar, de modo que todos los irónicos y no irónicos del fold se usan antes de repetir alguno. Durante el entrenamiento se comprueba, lote por lote, que la composición es 8 + 8.

**Comparación.**

- Principal: F1-Macro *out-of-fold* (OOF) en la **época del refit** de la configuración actual (`refit_epochs`), igual para todas las opciones.
- Contrastes pareados (mismos tweets, folds y semillas) con **IC 95 % por bootstrap de grupos** (se remuestrean `GROUP_ID`, no tweets sueltos) y, además, la diferencia en cada uno de los 9 folds (3 variantes × 3 folds).
- Por clase (F1, precisión y *recall* de irónico y no irónico), proporción de tweets predichos como irónicos, **brecha entrenamiento–validación** (F1 y NLL, con el train del fold evaluado en modo evaluación) y **calibración** (NLL y ECE, antes y después de un *temperature scaling* ajustado con los otros dos folds).

**Regla de decisión (fijada antes de ejecutar).** BBFL **merece confirmarse** solo si, frente a `REF`: (1) la diferencia media de las 3 variantes tiene IC 95 % por encima de 0; (2) la diferencia es positiva en al menos 2 de 3 variantes; y (3) es positiva en al menos 6 de los 9 folds. Aun así, el piloto **no cambia la configuración final**: solo indica si conviene una confirmación con Nested CV.

Costo: 5 opciones × 3 variantes × 3 folds = **45 fine-tunings**, más una evaluación del train del fold por época. Si se interrumpe, al volver a ejecutar se reutilizan los folds terminados.

**Referencias.** Singh, D., Liu, M., Schucany, W., et al. (2023). Batch-balanced focal loss: a hybrid solution to class imbalance in deep learning. *Journal of Medical Imaging*, 10(5), 051809. — Lin, T.-Y., Goyal, P., Girshick, R., He, K., y Dollár, P. (2017). Focal loss for dense object detection. *ICCV*, 2980–2988.

## 1. Librerías y entorno

In [ ]:
import os
import re

# Declarar antes de importar Transformers/Hugging Face para ocultar barras de descarga.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

import gc
import time
import json
import pickle
import html
import random
import tempfile
import shutil
import inspect
import warnings
from pathlib import Path
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import optuna

from torch.utils.data import Dataset, Subset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)
from IPython.display import display, HTML

import matplotlib

matplotlib.use("Agg")  # Evita abrir una ventana por gráfica durante cientos de fits.
import matplotlib.pyplot as plt

import transformers
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    set_seed,
)
from transformers.trainer_callback import PrinterCallback, ProgressCallback
from transformers.trainer_utils import get_last_checkpoint

try:
    from huggingface_hub.utils import disable_progress_bars

    disable_progress_bars()
except (ImportError, AttributeError):
    pass
transformers.logging.set_verbosity_error()
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)

VERSIONES = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "optuna": optuna.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
print("Versiones:", " | ".join(f"{k} {v}" for k, v in VERSIONES.items()))
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_fscore_support


## 2. Configuración del modelo (igual que `04c_RoBERTuito.ipynb`) y del piloto

In [ ]:
MODEL_NAME = "pysentimiento/robertuito-base-uncased"

DATA_DIR = Path("../../data/transformers")
RESULTS_DIR = Path("../../data/resultados_robertuito_grouped_5x3_tpe20")
PROGRESS_DIR = RESULTS_DIR / "seguimiento"

TEXT_COLS = {"PYSENTIMIENTO": "MESSAGE_TRANSFORMER"}
LABEL_COL = "IS_IRONIC"
ID_COL = "ID"
ORIGINAL_COL = "MESSAGE"
GROUP_COL = "GROUP_ID"

MAX_LENGTH = 128
SEED = 42

# ------------------------------------------------------------
# "prueba" valida rápidamente todo el pipeline.
# "completo" = experimento definitivo 5×3 / TPE20.
# ------------------------------------------------------------
MODO = "completo"  # primero "prueba"; luego "completo" y reiniciar kernel

if MODO == "prueba":
    VARIANTES = ["mx"]
    N_OUTER = 2
    N_INNER = 2
    N_TRIALS = 2
    N_STARTUP_TRIALS = 1
elif MODO == "completo":
    VARIANTES = ["mx", "es", "cu"]
    N_OUTER = 5
    N_INNER = 3
    N_TRIALS = 20
    N_STARTUP_TRIALS = 5
else:
    raise ValueError("MODO debe ser 'prueba' o 'completo'.")

# RUN_TAG corto + subcarpetas compactas para Windows.
RUN_TAG = "r5x3t20"
RUN_DIR = PROGRESS_DIR / RUN_TAG / MODO
TMP_DIR = RUN_DIR / "tmp"
CHECKPOINT_DIR = RUN_DIR / "ckpt"
FIT_STATE_DIR = RUN_DIR / "state"
CURVE_DIR = RUN_DIR / "curves"
GRAPH_DIR = RUN_DIR / "graphs"
AUDIT_DIR = RUN_DIR / "audit"

for carpeta in [
    RESULTS_DIR,
    PROGRESS_DIR,
    RUN_DIR,
    TMP_DIR,
    CHECKPOINT_DIR,
    FIT_STATE_DIR,
    CURVE_DIR,
    GRAPH_DIR,
    AUDIT_DIR,
]:
    carpeta.mkdir(parents=True, exist_ok=True)

# Flujo recomendado (sección 18): opcionalmente una corrida corta con MODO="prueba";
# luego MODO="completo". Nested CV y optimización final usan solo train y corren seguidas;
# el test oficial requiere además CONFIRMACION_TEST.
EJECUTAR_NESTED = True
EJECUTAR_OPTIMIZACION_FINAL = True  # solo train; se ejecuta tras la Nested CV
EJECUTAR_TEST_OFICIAL = False
CONFIRMACION_TEST = ""

SEMILLAS_FINALES = [42, 52, 62, 72, 82]

# Exportación del modelo final e interpretabilidad (SHAP)
SEMILLA_EXPORTACION = 42  # el modelo de esta semilla del test se guarda en modelo_final/
N_SHAP_POR_TIPO = 50  # tuits explicados por tipo (VP, VN, FP, FN) y variante
SHAP_BATCH = 32  # lote de inferencia de SHAP (bajar si falta memoria de GPU)
MIN_TEXTOS_TOKEN = 5  # un token entra al análisis global si aparece en ≥ 5 tuits explicados
TOP_TOKENS = 10  # tokens por sentido (irónico / no irónico) en la figura global
MAX_TOKENS_EJEMPLO = 40  # tokens mostrados en cada ejemplo local
CLEAN_COMPLETED_CHECKPOINTS = True

# Seguimiento: panel de progreso común a todos los notebooks de modelos.
NOMBRE_EXPERIMENTO = "RoBERTuito"
HEARTBEAT_SEGUNDOS = 30
# Pitido de Windows al terminar o detenerse una etapa larga (False para desactivarlo).
AVISO_SONORO = True

# Optuna TPE: 5 trials iniciales aleatorios y 15 adaptativos en modo completo.
# Espacio común a BETO, DistilBETO y RoBERTuito: se buscan weight_decay y classifier_dropout;
# el resto queda fijo (justificación en la celda «Decisiones de entrenamiento»).
SEARCH_SPACE = {
    "preprocessing": ["PYSENTIMIENTO"],
    "learning_rate": [2e-5],
    "batch_size": [16],
    "num_train_epochs": [6],  # máximo; la época efectiva la decide el early stopping
    "weight_decay": [0.01, 0.1],
    "loss_mode": ["balanced"],  # pérdida ponderada por clase (la ironía es minoritaria)
    "classifier_dropout": [0.1, 0.2, 0.3],
    "label_smoothing": [0.1],  # solo en la pérdida de entrenamiento
    "lr_decay": [0.9],  # LR por capas: cada capa inferior × 0.9
    "dropout_interno": [0.2],  # dropout oculto y de atención del encoder
    "congelar_frac": [0.0],  # se ajustan todas las capas
    "mascara_tokens": [0.0],  # fracción de tokens reemplazados por [MASK] solo al entrenar
}

# Warmup explícito. En transformers 5.x se declara con `warmup_steps` (fracción en [0, 1)).
WARMUP_FRACCION = 0.0

# Early stopping de los Inner Folds: épocas sin mejora de F1 antes de detener.
PACIENCIA_EARLY_STOPPING = 2

SEARCH_SPACE_REQUERIDO = {
    "preprocessing",
    "learning_rate",
    "batch_size",
    "num_train_epochs",
    "weight_decay",
    "loss_mode",
    "classifier_dropout",
    "label_smoothing",
    "lr_decay",
    "dropout_interno",
    "congelar_frac",
    "mascara_tokens",
}
faltantes_search_space = SEARCH_SPACE_REQUERIDO - set(SEARCH_SPACE)
if faltantes_search_space:
    raise KeyError(f"SEARCH_SPACE incompleto. Faltan: {sorted(faltantes_search_space)}")

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

fits_teoricos = len(VARIANTES) * N_OUTER * (N_TRIALS * N_INNER + 1)

print(f"[CONFIG][RoBERTuito] modo={MODO} | variantes={VARIANTES}")
print(
    f"[CONFIG][RoBERTuito] Nested={N_OUTER}×{N_INNER} | "
    f"trials={N_TRIALS} | startup={N_STARTUP_TRIALS}"
)
print(
    f"[CONFIG][RoBERTuito] fine-tunings teóricos={fits_teoricos} | "
    f"nested_activo={EJECUTAR_NESTED}"
)
print(f"[CONFIG][RoBERTuito] RUN_DIR={RUN_DIR.resolve()}")
print("[CONFIG][RoBERTuito] SEARCH_SPACE OK:", sorted(SEARCH_SPACE))

# Comprobación preventiva de longitud de ruta en Windows (peor caso del espacio).
_ruta_ckpt_ejemplo = (
    CHECKPOINT_DIR / "mx" / "outer1" / "P_l2_b16_e6_w10_b_d3_s1_r9_i2_c25_m15" / "inner3" / "checkpoint-99999"
).resolve()

print(
    f"[CONFIG][RoBERTuito] longitud ruta checkpoint ejemplo="
    f"{len(str(_ruta_ckpt_ejemplo))} caracteres"
)

if os.name == "nt" and len(str(_ruta_ckpt_ejemplo)) >= 230:
    raise RuntimeError(
        "La ruta prevista para checkpoints sigue siendo demasiado larga en Windows. "
        "Mueve el repositorio a una ruta más corta antes de entrenar."
    )

In [ ]:
# ============================================================
# CONFIGURACIÓN DEL PILOTO BBFL (solo train oficial; no toca el test)
# ============================================================
PILOTO_DIR = Path("../../data/pilotos_bbfl") / NOMBRE_EXPERIMENTO.lower()
PILOTO_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = PILOTO_DIR            # logs y auditorías del piloto, fuera de los resultados finales
TMP_DIR = PILOTO_DIR / "tmp"
TMP_DIR.mkdir(parents=True, exist_ok=True)

RUTA_CONFIG_FINAL = RESULTS_DIR / f"{NOMBRE_EXPERIMENTO.lower()}_config_final.json"
if not RUTA_CONFIG_FINAL.exists():
    raise FileNotFoundError(f"Falta {RUTA_CONFIG_FINAL}: ejecuta antes el notebook 04 de {NOMBRE_EXPERIMENTO}.")
CONFIG_FINAL = json.loads(RUTA_CONFIG_FINAL.read_text(encoding="utf-8"))

# pesos=True usa los pesos de clase "balanced" de la configuración actual.
OPCIONES = {
    "REF":    {"perdida": "ce",    "gamma": 0.0, "pesos": True,  "label_smoothing": 0.1, "lotes": "habituales",
               "nombre": "Actual: CE ponderada + LS 0.1"},
    "CE":     {"perdida": "ce",    "gamma": 0.0, "pesos": False, "label_smoothing": 0.0, "lotes": "habituales",
               "nombre": "CE sin pesos"},
    "FL":     {"perdida": "focal", "gamma": 2.0, "pesos": False, "label_smoothing": 0.0, "lotes": "habituales",
               "nombre": "Focal γ=2 sin pesos"},
    "CE_bal": {"perdida": "ce",    "gamma": 0.0, "pesos": False, "label_smoothing": 0.0, "lotes": "balanceados",
               "nombre": "CE sin pesos + lotes 8/8"},
    "BBFL":   {"perdida": "focal", "gamma": 2.0, "pesos": False, "label_smoothing": 0.0, "lotes": "balanceados",
               "nombre": "BBFL: Focal γ=2 + lotes 8/8"},
}
# Contrastes pareados: (nombre, opción A, opción B) -> se reporta A − B.
CONTRASTES = [
    ("BBFL vs. actual (principal)", "BBFL", "REF"),
    ("Balanceo con CE", "CE_bal", "CE"),
    ("Balanceo con Focal", "BBFL", "FL"),
    ("Focal con lotes habituales", "FL", "CE"),
    ("Focal con lotes balanceados", "BBFL", "CE_bal"),
    ("Pesos + LS de la config. actual", "REF", "CE"),
]
VARIANTES_PILOTO = ["mx", "es", "cu"]
N_FOLDS = 3
POR_CLASE = 8                   # 8 irónicos + 8 no irónicos = batch 16
N_BOOTSTRAP = 2000

for v in VARIANTES_PILOTO:
    c = CONFIG_FINAL[v]
    assert int(c["batch_size"]) == 2 * POR_CLASE, f"{v}: batch_size={c['batch_size']}, se esperaba {2 * POR_CLASE}"
print(f"[PILOTO BBFL][{NOMBRE_EXPERIMENTO}] salida = {PILOTO_DIR.resolve()}")
print(f"[PILOTO BBFL][{NOMBRE_EXPERIMENTO}] {len(OPCIONES)} opciones × {len(VARIANTES_PILOTO)} variantes × "
      f"{N_FOLDS} folds = {len(OPCIONES) * len(VARIANTES_PILOTO) * N_FOLDS} fine-tunings")
for v in VARIANTES_PILOTO:
    c = CONFIG_FINAL[v]
    print(f"  {v.upper()}: wd={c['weight_decay']} | classifier_dropout={c['classifier_dropout']} | "
          f"máscara={c.get('mascara_tokens', 0.0)} | épocas={c['num_train_epochs']} | refit={c['refit_epochs']}")

## 3. Datos, tokenizador y grupos (igual que el notebook 04)

In [ ]:
MANIFEST_PATH = DATA_DIR / "manifest_preprocesamiento_transformers.csv"
PIPELINE_VERSION_ESPERADA = "transformers_comun_v2"


def cargar_manifest_preprocesamiento():
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f"Falta {MANIFEST_PATH}. Ejecuta primero "
            "`01_preprocesamiento_dl_transformers.ipynb`."
        )

    manifest = pd.read_csv(MANIFEST_PATH)

    requeridas = {"pipeline_version", "variante", "train_final"}
    faltantes = requeridas - set(manifest.columns)
    if faltantes:
        raise ValueError(f"Manifest incompleto; faltan columnas: {sorted(faltantes)}")

    versiones = set(manifest["pipeline_version"].astype(str))
    if versiones != {PIPELINE_VERSION_ESPERADA}:
        raise ValueError(
            f"Versión de preprocessing inesperada: {versiones}. "
            f"Se esperaba {PIPELINE_VERSION_ESPERADA}."
        )

    return manifest


MANIFEST_PREPROCESAMIENTO = cargar_manifest_preprocesamiento()


def validar_archivo_train_con_manifest(variante):
    ruta = DATA_DIR / f"train_transformers_{variante}.csv"
    if not ruta.exists():
        raise FileNotFoundError(f"Falta {ruta}. Ejecuta primero el preprocessing Transformer.")

    fila = MANIFEST_PREPROCESAMIENTO[MANIFEST_PREPROCESAMIENTO["variante"].astype(str) == variante]
    if len(fila) != 1:
        raise ValueError(f"{variante}: se esperaba exactamente una fila en el manifest.")

    return ruta, fila.iloc[0]


def cargar_train(variante):
    ruta, manifest_row = validar_archivo_train_con_manifest(variante)
    df = pd.read_csv(ruta)

    requeridas = [ID_COL, ORIGINAL_COL, LABEL_COL, *TEXT_COLS.values()]
    faltantes = [c for c in requeridas if c not in df.columns]
    if faltantes:
        raise ValueError(f"{variante}: faltan columnas {faltantes}")

    df[ID_COL] = df[ID_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    for col in [ORIGINAL_COL, *TEXT_COLS.values()]:
        df[col] = df[col].fillna("").astype(str)

    if df[ID_COL].duplicated().any():
        raise ValueError(f"{variante}: IDs duplicados en el train FINAL.")

    if not set(df[LABEL_COL].unique()).issubset({0, 1}):
        raise ValueError(f"{variante}: etiquetas fuera de 0/1.")

    vacios = {
        col: int(df[col].eq("").sum())
        for col in [ORIGINAL_COL, *TEXT_COLS.values()]
        if df[col].eq("").any()
    }
    if vacios:
        raise ValueError(
            f"{variante}: entradas vacías en el train final {vacios}. "
            "Vuelve a ejecutar `01_preprocesamiento_dl_transformers.ipynb` "
            "(MESSAGE_TRANSFORMER requiere `pysentimiento` instalado)."
        )

    esperado = int(manifest_row["train_final"])
    if len(df) != esperado:
        raise ValueError(f"{variante}: train={len(df)} pero manifest={esperado}.")

    return df.reset_index(drop=True)


for variante in ["mx", "es", "cu"]:
    tmp = cargar_train(variante)
    print(
        f"[DATA][RoBERTuito][{variante.upper()}] "
        f"n={len(tmp)} | "
        f"clases={tmp[LABEL_COL].value_counts().sort_index().to_dict()} | "
        "manifest=OK"
    )

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Tokenizer:", tokenizer.__class__.__name__)
print("do_lower_case:", getattr(tokenizer, "do_lower_case", "gestionado por tokenizer"))
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer, pad_to_multiple_of=8 if torch.cuda.is_available() else None
)


class DatasetTokenizado(Dataset):
    def __init__(self, textos, etiquetas):
        self.encoded = tokenizer(
            [str(t) for t in textos], truncation=True, max_length=MAX_LENGTH, padding=False
        )
        self.labels = np.asarray(etiquetas, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encoded.items()}
        item["labels"] = int(self.labels[i])
        return item


def cache_dataset(df):
    y = df[LABEL_COL].to_numpy(dtype=np.int64)
    return {prep: DatasetTokenizado(df[col].tolist(), y) for prep, col in TEXT_COLS.items()}, y

In [ ]:
from collections import defaultdict


def colapsar_espacios(texto):
    return re.sub(r"\s+", " ", str(texto)).strip()


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def clave_tokens(ids):
    """Representación directa de los input_ids, sin hashes."""
    return ",".join(map(str, ids))


def construir_grupos_robertuito(df, variante, tokenizador=tokenizer, max_length=MAX_LENGTH):
    """Agrupa entradas equivalentes para evitar que crucen train/validación.

    Agrupación propia del modelo: texto original, cada representación de
    entrada (TEXT_COLS) y su secuencia tokenizada (truncada a MAX_LENGTH).
    """
    trabajo = df.copy().reset_index(drop=True)
    trabajo["KEY_ORIGINAL"] = trabajo[ORIGINAL_COL].map(colapsar_espacios)
    for prep, columna in TEXT_COLS.items():
        trabajo[f"KEY_{prep}"] = trabajo[columna].map(colapsar_espacios)

    for prep, columna in TEXT_COLS.items():
        ids = tokenizador(
            trabajo[columna].astype(str).tolist(),
            truncation=True,
            max_length=max_length,
            padding=False,
        )["input_ids"]
        trabajo[f"KEY_TOK_{prep}"] = [clave_tokens(x) for x in ids]

    representaciones = ["ORIGINAL", *TEXT_COLS, *[f"TOK_{p}" for p in TEXT_COLS]]
    key_cols = [f"KEY_{r}" for r in representaciones]
    uf = UnionFind(len(trabajo))

    for col in key_cols:
        first_seen = {}
        for idx, key in enumerate(trabajo[col].tolist()):
            if not key:
                continue
            if key in first_seen:
                uf.union(idx, first_seen[key])
            else:
                first_seen[key] = idx

    componentes = defaultdict(list)
    for idx in range(len(trabajo)):
        componentes[uf.find(idx)].append(idx)

    group_ids = [None] * len(trabajo)

    componentes_ordenados = sorted(componentes.values(), key=lambda miembros: min(miembros))

    for numero_grupo, miembros in enumerate(componentes_ordenados, start=1):
        gid = f"{variante}_G{numero_grupo:04d}"
        for idx in miembros:
            group_ids[idx] = gid
    trabajo[GROUP_COL] = group_ids

    resumen = []
    for nombre, col in zip(representaciones, key_cols):
        sizes = trabajo.groupby(col).size()
        repetidos = sizes[sizes > 1]
        resumen.append(
            {
                "variante": variante,
                "representacion": nombre,
                "filas_en_repeticiones": int(trabajo[col].isin(repetidos.index).sum()),
                "grupos_repetidos": int(len(repetidos)),
            }
        )

    gstats = trabajo.groupby(GROUP_COL)[LABEL_COL].agg(["size", "nunique"])
    conflictivos = set(gstats[(gstats["size"] > 1) & (gstats["nunique"] > 1)].index)
    detalle_conflictos = trabajo[trabajo[GROUP_COL].isin(conflictivos)].copy()

    resumen = pd.DataFrame(resumen)
    resumen["n_train"] = len(trabajo)
    resumen["n_group_id"] = trabajo[GROUP_COL].nunique()
    resumen["max_group_size"] = int(trabajo.groupby(GROUP_COL).size().max())
    resumen["grupos_con_labels_mixtos"] = len(conflictivos)
    return trabajo, resumen, detalle_conflictos


TRAIN_AGRUPADO = {}
AUDITORIA_GRUPOS = []
DETALLES_CONFLICTOS = []

# Se construyen los grupos de las tres variantes antes de una corrida larga.
for variante in ["mx", "es", "cu"]:
    df_audit = cargar_train(variante)
    agrupado, resumen, conflictos = construir_grupos_robertuito(df_audit, variante)
    TRAIN_AGRUPADO[variante] = agrupado
    AUDITORIA_GRUPOS.append(resumen)
    if not conflictos.empty:
        conflictos["variante"] = variante
        DETALLES_CONFLICTOS.append(conflictos)

df_auditoria_grupos = pd.concat(AUDITORIA_GRUPOS, ignore_index=True)
df_auditoria_grupos.to_csv(RUN_DIR / "auditoria_grupos_train.csv", index=False)
display(df_auditoria_grupos)

if DETALLES_CONFLICTOS:
    df_conflictos_grupo = pd.concat(DETALLES_CONFLICTOS, ignore_index=True)
    df_conflictos_grupo.to_csv(RUN_DIR / "auditoria_grupos_labels_mixtos.csv", index=False)

for variante in ["mx", "es", "cu"]:
    df_g = TRAIN_AGRUPADO[variante]
    n_mixtos = int(df_g.groupby(GROUP_COL)[LABEL_COL].nunique().gt(1).sum())
    print(
        f"[AUDIT][RoBERTuito][{variante.upper()}] n={len(df_g)} | "
        f"groups={df_g[GROUP_COL].nunique()} | "
        f"max_group={df_g.groupby(GROUP_COL).size().max()} | labels_mixtos={n_mixtos}"
    )

    if n_mixtos > 0:
        print(
            f"  ADVERTENCIA: {n_mixtos} GROUP_ID contienen etiquetas mixtas. "
            "No es data leakage porque el grupo permanece íntegro dentro de un fold, "
            "pero representa ambigüedad/ruido de etiqueta y debe documentarse."
        )

### 3.1 Modelo, Trainer y argumentos de entrenamiento (igual que el notebook 04)

In [ ]:
def establecer_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def liberar_memoria():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def softmax_np(logits):
    z = np.asarray(logits, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def nll_binaria(probs, labels):
    """NLL estándar: SIN pesos de clase y SIN label smoothing."""
    labels = np.asarray(labels, dtype=int)
    return float(-np.mean(np.log(np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1.0))))


def ece_binaria(probs, labels, n_bins=10):
    """ECE con 10 bins de igual masa sobre la confianza de la clase predicha."""
    labels = np.asarray(labels, dtype=int)
    conf = probs.max(axis=1)
    acierto = (probs.argmax(axis=1) == labels).astype(float)
    grupos = np.array_split(np.argsort(conf, kind="stable"), n_bins)
    return float(sum(len(g) / len(conf) * abs(acierto[g].mean() - conf[g].mean())
                     for g in grupos if len(g)))


def metricas(prediction):
    logits, labels = prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    y_pred = np.asarray(logits).argmax(axis=-1)
    probs = softmax_np(logits)
    return {
        "nll": nll_binaria(probs, labels),
        "ece": ece_binaria(probs, labels),
        "f1_macro": f1_score(labels, y_pred, average="macro"),
        "accuracy": accuracy_score(labels, y_pred),
        "precision_macro": precision_score(labels, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, y_pred, average="macro", zero_division=0),
    }


def pesos_balanceados(y):
    classes = np.array([0, 1])
    w = compute_class_weight("balanced", classes=classes, y=np.asarray(y))
    return torch.tensor(w, dtype=torch.float32)


PATRON_CAPA = re.compile(r"\.layer\.(\d+)\.")


def congelar_fraccion_inferior(model, fraccion):
    """congela embeddings + la fracción inferior de capas del encoder."""
    if not fraccion:
        return []
    n_total = 1 + max(int(m.group(1)) for n, _ in model.named_parameters()
                      if (m := PATRON_CAPA.search(n)))
    n_congeladas = max(1, round(n_total * float(fraccion)))
    capas = set()
    for nombre, p in model.named_parameters():
        m = PATRON_CAPA.search(nombre)
        if "embeddings" in nombre or (m and int(m.group(1)) < n_congeladas):
            p.requires_grad = False
            if m:
                capas.add(int(m.group(1)))
    return sorted(capas)


def crear_modelo(seed, classifier_dropout=0.2, cfg=None):
    """`cfg` aporta `dropout_interno` y `congelar_frac`."""
    cfg = cfg or {}
    establecer_semilla(seed)
    with (RUN_DIR / "carga_modelos.log").open("a", encoding="utf-8") as log, redirect_stdout(
        log
    ), redirect_stderr(log):
        config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
        if config.model_type != "roberta":
            raise ValueError(f"Se esperaba arquitectura RoBERTa, recibida: {config.model_type}")
        # RoBERTa: dropout de la cabeza de clasificación.
        config.classifier_dropout = float(classifier_dropout)
        # dropout interno (oculto y de atención) del encoder y los embeddings.
        dropout_interno = float(cfg.get("dropout_interno", getattr(config, "hidden_dropout_prob")))
        config.hidden_dropout_prob = dropout_interno
        config.attention_probs_dropout_prob = dropout_interno
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
    congelar_fraccion_inferior(model, cfg.get("congelar_frac", 0.0))
    return model


class ClasificadorTrainer(Trainer):
    """pérdida ponderada opcional + label smoothing (solo en train) + LR decay por capas
    + enmascarado aleatorio de tokens."""

    def __init__(self, *args, class_weights=None, label_smoothing=0.0, lr_decay=1.0,
                 mascara_tokens=0.0, **kwargs):
        self.class_weights = class_weights
        self.label_smoothing = float(label_smoothing)
        self.lr_decay = float(lr_decay)
        self.mascara_tokens = float(mascara_tokens)
        self._especiales = torch.tensor(sorted(set(tokenizer.all_special_ids)))
        super().__init__(*args, **kwargs)

    def _enmascarar(self, inputs):
        """Reemplaza al azar `mascara_tokens` de los tokens (no especiales, no padding) por el
        token de máscara. Validación, refit-evaluación y test leen siempre el texto completo."""
        ids = inputs["input_ids"].clone()
        candidatos = ~torch.isin(ids, self._especiales.to(ids.device))
        if "attention_mask" in inputs:
            candidatos &= inputs["attention_mask"].bool()
        sorteo = torch.rand(ids.shape, device=ids.device) < self.mascara_tokens
        ids[candidatos & sorteo] = tokenizer.mask_token_id
        return {**inputs, "input_ids": ids}

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        if model.training and self.mascara_tokens > 0:
            inputs = self._enmascarar(inputs)
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        weights = (
            self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        )
        # Smoothing SOLO al entrenar. En evaluación la pérdida (eval_loss) sigue ponderada por clase;
        # la NLL sin pesos de clase se calcula aparte en metricas().
        ls = self.label_smoothing if model.training else 0.0
        loss = F.cross_entropy(outputs.logits, labels, weight=weights, label_smoothing=ls)
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self, *args, **kwargs):
        if self.optimizer is not None:
            return self.optimizer
        if self.lr_decay >= 1.0:
            return super().create_optimizer(*args, **kwargs)
        # Capas: BERT/RoBERTa `encoder.layer.N`, DistilBERT `transformer.layer.N`.
        patron = re.compile(r"\.layer\.(\d+)\.")
        n_capas = 1 + max(
            int(m.group(1)) for n, _ in self.model.named_parameters() if (m := patron.search(n))
        )
        base_lr, wd = self.args.learning_rate, self.args.weight_decay
        grupos = {}
        for nombre, p in self.model.named_parameters():
            if not p.requires_grad:
                continue
            m = patron.search(nombre)
            if m:
                profundidad = n_capas - int(m.group(1))  # última capa = 1
            elif "embeddings" in nombre:
                profundidad = n_capas + 1  # embeddings = LR más bajo
            else:
                profundidad = 0  # cabeza de clasificación / pooler = LR base
            sin_wd = nombre.endswith("bias") or "LayerNorm" in nombre or "layer_norm" in nombre
            clave = (base_lr * self.lr_decay**profundidad, 0.0 if sin_wd else wd)
            grupos.setdefault(clave, []).append(p)
        self.optimizer = torch.optim.AdamW(
            [{"params": ps, "lr": lr, "weight_decay": w} for (lr, w), ps in grupos.items()]
        )
        return self.optimizer


class ParadaTemprana(TrainerCallback):
    """Early stopping por F1-Macro de validación sin guardar checkpoints (Inner Folds)."""

    def __init__(self, paciencia=2):
        self.paciencia, self.mejor, self.sin_mejora = int(paciencia), -np.inf, 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        f1 = (metrics or {}).get("eval_f1_macro")
        if f1 is None:
            return control
        if f1 > self.mejor:
            self.mejor, self.sin_mejora = f1, 0
        else:
            self.sin_mejora += 1
            if self.sin_mejora >= self.paciencia:
                control.should_training_stop = True
        return control


class DetenerEnEpoca(TrainerCallback):
    """Refit/test con el MISMO scheduler que la búsqueda: se detiene en la época elegida."""

    def __init__(self, epocas):
        self.epocas = int(epocas)

    def on_epoch_end(self, args, state, control, **kwargs):
        if state.epoch is not None and state.epoch >= self.epocas - 1e-6:
            control.should_training_stop = True
        return control


In [ ]:
def argumentos_entrenamiento(carpeta, cfg, seed, con_validacion=True, epochs=None):
    params = {
        "output_dir": str(carpeta),
        "learning_rate": cfg["learning_rate"],
        "per_device_train_batch_size": cfg["batch_size"],
        "per_device_eval_batch_size": cfg["batch_size"],
        "num_train_epochs": epochs if epochs is not None else cfg["num_train_epochs"],
        "weight_decay": cfg["weight_decay"],
        "optim": "adamw_torch",
        # Trainer recoge loss por época en state.log_history;
        # PrinterCallback se elimina para no saturar VS Code.
        "logging_strategy": "epoch",
        "save_strategy": "epoch",  # Inner y refit: reinicio desde última época completa.
        "load_best_model_at_end": bool(con_validacion),
        "save_total_limit": 2,  # último + mejor (si son diferentes)
        "save_only_model": False,  # Estado optimizer/scheduler requerido para reanudar.
        "report_to": "none",
        "disable_tqdm": True,
        # Necesario para que EarlyStoppingCallback conserve su contador
        # al reanudar desde un checkpoint.
        "restore_callback_states_from_checkpoint": True,
        "dataloader_num_workers": 0,
        "seed": seed,
        "data_seed": seed,
        "bf16": USE_BF16,
        "fp16": USE_FP16,
    }
    if con_validacion:
        # Inner Folds sin checkpoints: el modelo interno se descarta y F1/mejor época
        # se leen de la curva. El early stopping lo hace ParadaTemprana.
        params.update({"save_strategy": "no", "load_best_model_at_end": False})
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        params["eval_strategy"] = "epoch" if con_validacion else "no"
    elif "evaluation_strategy" in sig:
        params["evaluation_strategy"] = "epoch" if con_validacion else "no"
    else:
        raise RuntimeError("Esta versión de Transformers no admite evaluación por época.")
    # warmup explícito (transformers 5: `warmup_steps` acepta una fracción en [0, 1)).
    if "warmup_ratio" in sig:
        params["warmup_ratio"] = float(WARMUP_FRACCION)
    else:
        params["warmup_steps"] = float(WARMUP_FRACCION)
    # nunca descartar argumentos en silencio.
    descartados = sorted(k for k in params if k not in sig)
    if descartados:
        raise RuntimeError(f"TrainingArguments no acepta {descartados} en esta versión de transformers.")
    return TrainingArguments(**params)


def mejor_epoca(trainer, max_epochs):
    evaluaciones = [x for x in trainer.state.log_history if "eval_f1_macro" in x and "epoch" in x]
    if not evaluaciones:
        raise RuntimeError("No se encontró F1 de validación por época.")
    mejor = max(evaluaciones, key=lambda x: x["eval_f1_macro"])
    return int(np.clip(round(float(mejor["epoch"])), 1, max_epochs))


def predecir(trainer, dataset):
    logits = trainer.predict(dataset).predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    return np.argmax(logits, axis=-1)

In [ ]:
def avisar_fin(ok=True):
    """Pitido de Windows al terminar (3 veces) o al detenerse (1 vez); en otros sistemas no hace nada."""
    if not AVISO_SONORO:
        return
    try:
        import winsound

        for _ in range(3 if ok else 1):
            winsound.MessageBeep(winsound.MB_ICONASTERISK if ok else winsound.MB_ICONHAND)
            time.sleep(0.5)
    except Exception:
        pass


def entrenador_silencioso(trainer):
    trainer.remove_callback(PrinterCallback)
    trainer.remove_callback(ProgressCallback)
    return trainer


def csv_atomico(df, ruta):
    ruta = Path(ruta)
    tmp = ruta.with_name(ruta.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, ruta)

## 4. Focal Loss, lotes balanceados y entrenamiento

`TrainerBBFL` hereda el Trainer del notebook 04 (LR por capas, máscara de tokens) y solo cambia, según la opción, la pérdida de entrenamiento y el armado de los lotes. La celda comprueba antes que Focal Loss con γ = 0 es idéntica a la entropía cruzada y que el muestreador produce lotes 8 + 8 que recorren todos los irónicos antes de repetirlos.

In [ ]:
def perdida_focal(logits, labels, gamma, weights=None, label_smoothing=0.0):
    """Focal Loss (Lin et al., 2017). Con gamma = 0 es exactamente F.cross_entropy
    (con los mismos pesos y label smoothing)."""
    logp = F.log_softmax(logits.float(), dim=-1)
    k = logp.size(-1)
    objetivo = F.one_hot(labels, k).float()
    if label_smoothing > 0:
        objetivo = objetivo * (1.0 - label_smoothing) + label_smoothing / k
    modulador = (1.0 - logp.exp()).clamp(min=0.0) ** gamma
    por_clase = -objetivo * modulador * logp
    if weights is not None:
        w = weights.float().to(logp.device)
        return (por_clase * w).sum(dim=-1).sum() / w[labels].sum()
    return por_clase.sum(dim=-1).mean()


class MuestreadorBalanceado:
    """Lotes con exactamente `por_clase` ejemplos de cada clase (Singh et al., 2023).

    Cada clase se recorre en orden aleatorio sin repetir hasta agotarse y luego se vuelve a barajar.
    El número de lotes por época es el mismo que con lotes habituales (presupuesto comparable).
    """

    def __init__(self, etiquetas, por_clase, n_lotes, semilla):
        etiquetas = np.asarray(etiquetas, dtype=int)
        self.indices = {c: np.flatnonzero(etiquetas == c) for c in (0, 1)}
        for c, idx in self.indices.items():
            if len(idx) < por_clase:
                raise ValueError(f"La clase {c} tiene {len(idx)} ejemplos (< {por_clase}).")
        self.por_clase, self.n_lotes = int(por_clase), int(n_lotes)
        self.rng = np.random.default_rng(semilla)
        self.colas = {0: [], 1: []}

    def _tomar(self, clase):
        tomados = []
        while len(tomados) < self.por_clase:
            if not self.colas[clase]:
                self.colas[clase] = self.rng.permutation(self.indices[clase]).tolist()
            candidato = self.colas[clase].pop()
            if candidato in tomados:  # solo puede ocurrir al rebarajar; se deja para el siguiente lote
                self.colas[clase].insert(0, candidato)
                continue
            tomados.append(candidato)
        return tomados

    def __iter__(self):
        for _ in range(self.n_lotes):
            lote = self._tomar(0) + self._tomar(1)
            self.rng.shuffle(lote)
            yield [int(i) for i in lote]

    def __len__(self):
        return self.n_lotes


class TrainerBBFL(ClasificadorTrainer):
    """Trainer del notebook 04 con dos cambios opcionales: Focal Loss y lotes balanceados.

    La evaluación (validación y train del fold) usa siempre entropía cruzada sin label smoothing;
    la NLL y el ECE de `metricas()` no usan pesos de clase, así que son comparables entre opciones.
    """

    def __init__(self, *args, perdida="ce", gamma=0.0, muestreador=None, **kwargs):
        self.perdida = perdida
        self.gamma = float(gamma)
        self.muestreador = muestreador
        self.composicion_lotes = []
        super().__init__(*args, **kwargs)

    def get_train_dataloader(self):
        if self.muestreador is None:
            return super().get_train_dataloader()
        cargador = DataLoader(self.train_dataset, batch_sampler=self.muestreador, collate_fn=self.data_collator,
                              num_workers=self.args.dataloader_num_workers,
                              pin_memory=self.args.dataloader_pin_memory)
        return self.accelerator.prepare(cargador)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        if model.training and self.muestreador is not None:
            n1 = int(inputs["labels"].sum().item())
            self.composicion_lotes.append((len(inputs["labels"]) - n1, n1))
        if self.perdida == "ce" or not model.training:
            return super().compute_loss(model, inputs, return_outputs=return_outputs,
                                        num_items_in_batch=num_items_in_batch, **kwargs)
        if self.mascara_tokens > 0:
            inputs = self._enmascarar(inputs)
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        loss = perdida_focal(outputs.logits, labels, self.gamma, self.class_weights, self.label_smoothing)
        return (loss, outputs) if return_outputs else loss


# Comprobaciones previas.
_g = torch.Generator().manual_seed(0)
_z, _y, _w = torch.randn(64, 2, generator=_g), torch.randint(0, 2, (64,), generator=_g), torch.tensor([0.75, 1.5])
for _ls in (0.0, 0.1):
    for _peso in (None, _w):
        assert torch.allclose(perdida_focal(_z, _y, 0.0, _peso, _ls), F.cross_entropy(_z, _y, weight=_peso, label_smoothing=_ls), atol=1e-6)
_m = MuestreadorBalanceado(np.r_[np.zeros(1066), np.ones(533)].astype(int), POR_CLASE, 100, 0)
_lotes = list(_m) + list(_m)
_etq = np.r_[np.zeros(1066), np.ones(533)]
assert len(_m) == 100 and all(len(l) == 2 * POR_CLASE and len(set(l)) == len(l) and _etq[l].sum() == POR_CLASE for l in _lotes)
_vistos = np.concatenate(_lotes)
assert set(_vistos[_etq[_vistos] == 1]) == set(range(1066, 1599)), "los irónicos deben recorrerse completos antes de repetirse"
print("Focal Loss (γ=0 = CE) y muestreador balanceado (8 + 8 por lote, sin repetir dentro del lote) verificados.")

### 4.1 Ejecutar los 45 fine-tunings

Orden: variante → opción → fold. Cada fine-tuning evalúa por época la validación y el train del fold (para la brecha). Cada fold terminado se guarda (`.npz` + `.csv`); al volver a ejecutar se reutiliza.

In [ ]:
def splits_variante(variante):
    df = TRAIN_AGRUPADO[variante].copy()
    cache, y = cache_dataset(df)
    groups = df[GROUP_COL].astype(str).to_numpy()
    skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    return cache, y, groups, list(skf.split(np.zeros(len(y)), y, groups=groups))


def entrenar_fold(variante, opcion, numero, cache, y, tr, val):
    ruta = PILOTO_DIR / f"{opcion}_{variante}_fold{numero}.npz"
    if ruta.exists():
        return "reutilizado", None
    cfg = {k: v for k, v in CONFIG_FINAL[variante].items() if k != "refit_epochs"}
    op = OPCIONES[opcion]
    epocas = int(cfg["num_train_epochs"])
    ds = cache[cfg["preprocessing"]]
    seed_fold = SEED + SEED * 100 + numero          # igual en las 5 opciones
    carpeta = TMP_DIR / f"{opcion}_{variante}_f{numero}"
    n_lotes = int(np.ceil(len(tr) / int(cfg["batch_size"])))   # mismos pasos que con lotes habituales
    muestreador = (MuestreadorBalanceado(y[tr], POR_CLASE, n_lotes, seed_fold)
                   if op["lotes"] == "balanceados" else None)
    registro = {len(val): [], len(tr): []}

    def _metricas(prediction):
        logits = prediction[0][0] if isinstance(prediction[0], tuple) else prediction[0]
        registro[len(logits)].append(np.asarray(logits, dtype=np.float32))
        return metricas(prediction)

    model = crear_modelo(seed_fold, cfg["classifier_dropout"], cfg)   # checkpoint preentrenado
    trainer = None
    try:
        trainer = entrenador_silencioso(TrainerBBFL(
            model=model,
            args=argumentos_entrenamiento(carpeta, cfg, seed_fold, con_validacion=True, epochs=epocas),
            train_dataset=Subset(ds, tr.tolist()),
            eval_dataset={"val": Subset(ds, val.tolist()), "train": Subset(ds, tr.tolist())},
            data_collator=data_collator,
            compute_metrics=_metricas,
            class_weights=pesos_balanceados(y[tr]) if op["pesos"] else None,
            label_smoothing=op["label_smoothing"],
            lr_decay=cfg.get("lr_decay", 1.0),
            mascara_tokens=cfg.get("mascara_tokens", 0.0),
            perdida=op["perdida"], gamma=op["gamma"], muestreador=muestreador,
        ))
        trainer.train()
        if muestreador is not None:
            comp = np.array(trainer.composicion_lotes)
            if len(comp) != n_lotes * epocas or not (comp == POR_CLASE).all():
                raise RuntimeError(f"{opcion}/{variante}/fold{numero}: lotes no balanceados {np.unique(comp, axis=0)}")
        historia = trainer.state.log_history
        ev = [e for e in historia if "eval_val_f1_macro" in e]
        et = [e for e in historia if "eval_train_f1_macro" in e]
        if not (len(ev) == len(et) == len(registro[len(val)]) == len(registro[len(tr)]) == epocas):
            raise RuntimeError(f"{opcion}/{variante}/fold{numero}: evaluaciones incompletas")
        perdidas = {int(round(e["epoch"])): e["loss"] for e in historia if "loss" in e and "epoch" in e}
        tabla = pd.DataFrame({
            "opcion": opcion, "variante": variante, "fold": numero,
            "epoca": [int(round(e["epoch"])) for e in ev],
            "val_f1": [e["eval_val_f1_macro"] for e in ev],
            "val_nll": [e["eval_val_nll"] for e in ev], "val_ece": [e["eval_val_ece"] for e in ev],
            "train_f1": [e["eval_train_f1_macro"] for e in et],
            "train_nll": [e["eval_train_nll"] for e in et],
            "perdida_entrenamiento": [perdidas.get(int(round(e["epoch"])), np.nan) for e in ev],
            "lotes_por_epoca": n_lotes,
        })
        csv_atomico(tabla, PILOTO_DIR / f"{opcion}_{variante}_fold{numero}.csv")
        tmp = ruta.with_name(ruta.stem + "_tmp.npz")
        np.savez_compressed(tmp, logits=np.stack(registro[len(val)]), labels=y[val], val_idx=np.asarray(val),
                            epocas=tabla["epoca"].to_numpy())
        os.replace(tmp, ruta)  # el .npz final solo existe si el fold terminó completo
        return "entrenado", (None if muestreador is None else len(comp))
    finally:
        if trainer is not None:
            del trainer
        del model
        liberar_memoria()
        shutil.rmtree(carpeta, ignore_errors=True)


def ejecutar_piloto():
    total = len(OPCIONES) * len(VARIANTES_PILOTO) * N_FOLDS
    hechos, t_inicio = 0, time.time()
    for variante in VARIANTES_PILOTO:
        cache, y, _, splits = splits_variante(variante)
        refit = int(CONFIG_FINAL[variante]["refit_epochs"])
        for opcion in OPCIONES:
            for numero, (tr, val) in enumerate(splits, start=1):
                t0 = time.time()
                estado, n_lotes_bal = entrenar_fold(variante, opcion, numero, cache, y, tr, val)
                hechos += 1
                d = pd.read_csv(PILOTO_DIR / f"{opcion}_{variante}_fold{numero}.csv").set_index("epoca")
                extra = f" | lotes 8/8 verificados: {n_lotes_bal}" if n_lotes_bal else ""
                print(f"[{hechos:>2}/{total}] {variante.upper()} | {opcion:<6} | fold {numero} | {estado:<11} | "
                      f"F1 val (época {refit}) = {d.loc[refit, 'val_f1']:.4f} | train = {d.loc[refit, 'train_f1']:.4f} | "
                      f"{time.time() - t0:4.0f} s | total {(time.time() - t_inicio) / 60:5.1f} min{extra}", flush=True)
    avisar_fin(True)


ejecutar_piloto()

## 5. Resultados

### 5.1 F1-Macro, contrastes pareados e interacción

In [ ]:
def f1_desde_conteos(tp, fp, fn, tn):
    f1_1 = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros_like(tp, dtype=float), where=(2 * tp + fp + fn) > 0)
    f1_0 = np.divide(2 * tn, 2 * tn + fn + fp, out=np.zeros_like(tn, dtype=float), where=(2 * tn + fn + fp) > 0)
    return (f1_0 + f1_1) / 2


def conteos(y, p):
    return np.stack([(p == 1) & (y == 1), (p == 1) & (y == 0), (p == 0) & (y == 1), (p == 0) & (y == 0)], axis=1).astype(float)


def temperatura_optima(logits, y):
    rejilla = np.exp(np.linspace(np.log(0.25), np.log(8.0), 300))
    nll = [nll_binaria(softmax_np(logits / t), y) for t in rejilla]
    return float(rejilla[int(np.argmin(nll))])


def cargar_oof(variante, opcion):
    partes = [np.load(PILOTO_DIR / f"{opcion}_{variante}_fold{n}.npz") for n in range(1, N_FOLDS + 1)]
    n = len(TRAIN_AGRUPADO[variante])
    epocas = partes[0]["epocas"]
    logits = np.full((len(epocas), n, 2), np.nan, dtype=np.float32)
    y, fold = np.full(n, -1), np.zeros(n, dtype=int)
    for k, d in enumerate(partes, start=1):
        logits[:, d["val_idx"], :] = d["logits"]
        y[d["val_idx"]], fold[d["val_idx"]] = d["labels"], k
    assert (y >= 0).all() and not np.isnan(logits).any(), f"{variante}/{opcion}: OOF incompleto"
    return epocas, logits, y, fold


rng = np.random.default_rng(SEED)
FILAS, CURVAS, PRED, FOLDS_F1, BOOT = [], [], {}, [], {}
for variante in VARIANTES_PILOTO:
    refit = int(CONFIG_FINAL[variante]["refit_epochs"])
    grupos = TRAIN_AGRUPADO[variante][GROUP_COL].astype(str).to_numpy()
    _, inv = np.unique(grupos, return_inverse=True)
    G = inv.max() + 1
    pesos_boot = rng.multinomial(G, np.full(G, 1 / G), size=N_BOOTSTRAP).astype(float)   # bootstrap de grupos
    for opcion in OPCIONES:
        epocas, logits, y, fold = cargar_oof(variante, opcion)
        k = int(np.where(epocas == refit)[0][0])
        z = logits[k]
        probs = softmax_np(z)
        pred = probs.argmax(axis=1)
        PRED[(variante, opcion)] = (y, pred, fold)
        # temperatura ajustada con los otros 2 folds (sin mirar el fold evaluado)
        probs_t = np.zeros_like(probs)
        for f in range(1, N_FOLDS + 1):
            t = temperatura_optima(z[fold != f], y[fold != f])
            probs_t[fold == f] = softmax_np(z[fold == f] / t)
        por_fold = pd.concat([pd.read_csv(PILOTO_DIR / f"{opcion}_{variante}_fold{n}.csv") for n in range(1, N_FOLDS + 1)])
        r = por_fold[por_fold["epoca"] == refit]
        for _, f in r.iterrows():
            FOLDS_F1.append({"variante": variante, "opcion": opcion, "fold": int(f["fold"]), "val_f1": f["val_f1"]})
        p_cls, r_cls, f_cls, _ = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
        media_epoca = por_fold.groupby("epoca")["val_f1"].mean()
        FILAS.append({
            "variante": variante, "opcion": opcion, "epoca_refit": refit,
            "f1_macro": f1_score(y, pred, average="macro"),
            "f1_ironico": f_cls[1], "precision_ironico": p_cls[1], "recall_ironico": r_cls[1],
            "f1_no_ironico": f_cls[0], "precision_no_ironico": p_cls[0], "recall_no_ironico": r_cls[0],
            "prop_predicha_ironica": pred.mean(), "prop_real_ironica": y.mean(),
            "train_f1": r["train_f1"].mean(), "val_f1_folds": r["val_f1"].mean(),
            "brecha_f1": r["train_f1"].mean() - r["val_f1"].mean(),
            "train_nll": r["train_nll"].mean(), "val_nll_folds": r["val_nll"].mean(),
            "brecha_nll": r["val_nll"].mean() - r["train_nll"].mean(),
            "nll": nll_binaria(probs, y), "ece": ece_binaria(probs, y),
            "nll_temperatura": nll_binaria(probs_t, y), "ece_temperatura": ece_binaria(probs_t, y),
            "mejor_epoca_media": int(media_epoca.idxmax()), "f1_mejor_epoca_folds": media_epoca.max(),
        })
        for e, le in zip(epocas, logits):
            CURVAS.append({"variante": variante, "opcion": opcion, "epoca": int(e),
                           "f1_oof": f1_score(y, softmax_np(le).argmax(axis=1), average="macro")})
        # conteos por grupo para el bootstrap de grupos
        C = np.zeros((G, 4))
        np.add.at(C, inv, conteos(y, pred))
        BOOT[(variante, opcion)] = f1_desde_conteos(*(pesos_boot @ C).T)

RESUMEN = pd.DataFrame(FILAS)
CURVAS = pd.DataFrame(CURVAS)
FOLDS_F1 = pd.DataFrame(FOLDS_F1)

filas_c = []
for nombre, a, b in CONTRASTES:
    difs_var, boots = [], []
    for variante in VARIANTES_PILOTO:
        fa = RESUMEN.set_index(["variante", "opcion"]).loc[(variante, a), "f1_macro"]
        fb = RESUMEN.set_index(["variante", "opcion"]).loc[(variante, b), "f1_macro"]
        boot = BOOT[(variante, a)] - BOOT[(variante, b)]
        ff = FOLDS_F1.pivot_table(index=["variante", "fold"], columns="opcion", values="val_f1").loc[variante]
        difs_var.append(fa - fb)
        boots.append(boot)
        filas_c.append({"contraste": nombre, "A": a, "B": b, "variante": variante, "dif_A_menos_B": fa - fb,
                        "ic95_inf": np.percentile(boot, 2.5), "ic95_sup": np.percentile(boot, 97.5),
                        "folds_A_mejor": f"{int((ff[a] > ff[b]).sum())}/{N_FOLDS}"})
    boot_media = np.mean(boots, axis=0)  # bootstrap estratificado por variante
    ff_all = FOLDS_F1.pivot_table(index=["variante", "fold"], columns="opcion", values="val_f1")
    filas_c.append({"contraste": nombre, "A": a, "B": b, "variante": "media 3 variantes",
                    "dif_A_menos_B": float(np.mean(difs_var)),
                    "ic95_inf": np.percentile(boot_media, 2.5), "ic95_sup": np.percentile(boot_media, 97.5),
                    "folds_A_mejor": f"{int((ff_all[a] > ff_all[b]).sum())}/{len(ff_all)}"})
CONTRASTES_TAB = pd.DataFrame(filas_c)
# Interacción del diseño 2×2: ¿el balanceo ayuda distinto con Focal que con CE?
inter = []
for variante in VARIANTES_PILOTO:
    f = RESUMEN.set_index(["variante", "opcion"])["f1_macro"]
    b = (BOOT[(variante, "BBFL")] - BOOT[(variante, "FL")]) - (BOOT[(variante, "CE_bal")] - BOOT[(variante, "CE")])
    inter.append({"variante": variante,
                  "interaccion": (f[(variante, "BBFL")] - f[(variante, "FL")]) - (f[(variante, "CE_bal")] - f[(variante, "CE")]),
                  "ic95_inf": np.percentile(b, 2.5), "ic95_sup": np.percentile(b, 97.5)})
INTERACCION = pd.DataFrame(inter)

base = NOMBRE_EXPERIMENTO.lower()
csv_atomico(RESUMEN, PILOTO_DIR / f"{base}_bbfl_resumen.csv")
csv_atomico(CONTRASTES_TAB, PILOTO_DIR / f"{base}_bbfl_contrastes.csv")
csv_atomico(INTERACCION, PILOTO_DIR / f"{base}_bbfl_interaccion.csv")
csv_atomico(FOLDS_F1, PILOTO_DIR / f"{base}_bbfl_f1_por_fold.csv")
csv_atomico(CURVAS, PILOTO_DIR / f"{base}_bbfl_curvas.csv")

orden = list(OPCIONES)
tabla = RESUMEN.pivot(index="opcion", columns="variante", values="f1_macro").reindex(index=orden, columns=VARIANTES_PILOTO)
tabla["media"] = tabla.mean(axis=1)
print("1) F1-Macro OOF en la época del refit (train oficial, 3 folds agrupados):")
display(tabla.round(4))
print("\n2-3) Contrastes pareados (A − B), IC 95 % por bootstrap de grupos y folds en que A supera a B:")
display(CONTRASTES_TAB.round(4))
print("\nInteracción balanceo × función de error [(BBFL − FL) − (CE_bal − CE)]:")
display(INTERACCION.round(4))

### 5.2 Clases, brecha entrenamiento–validación y calibración

In [ ]:
cols_clase = ["f1_ironico", "precision_ironico", "recall_ironico", "f1_no_ironico", "precision_no_ironico",
              "recall_no_ironico", "prop_predicha_ironica", "prop_real_ironica"]
cols_brecha = ["train_f1", "val_f1_folds", "brecha_f1", "train_nll", "val_nll_folds", "brecha_nll"]
cols_cal = ["nll", "ece", "nll_temperatura", "ece_temperatura", "mejor_epoca_media", "f1_mejor_epoca_folds"]
for titulo, cols in [("4a) Por clase", cols_clase), ("4b) Brecha entrenamiento–validación (época del refit)", cols_brecha),
                     ("4c) Calibración (antes y después de temperature scaling con los otros folds) y mejor época", cols_cal)]:
    print(titulo)
    display(RESUMEN.set_index(["variante", "opcion"])[cols].reindex(
        [(v, o) for v in VARIANTES_PILOTO for o in OPCIONES]).round(4))

### 5.3 Figura

In [ ]:
fig, axes = plt.subplots(1, len(VARIANTES_PILOTO) + 1, figsize=(4.6 * (len(VARIANTES_PILOTO) + 1), 3.9), squeeze=False)
estilos = {"REF": ("k", "-"), "CE": ("C0", "--"), "FL": ("C3", "--"), "CE_bal": ("C0", ":"), "BBFL": ("C3", "-")}
for ax, variante in zip(axes[0], VARIANTES_PILOTO):
    for opcion in OPCIONES:
        d = CURVAS[(CURVAS["variante"] == variante) & (CURVAS["opcion"] == opcion)]
        color, linea = estilos[opcion]
        ax.plot(d["epoca"], d["f1_oof"], color=color, linestyle=linea, marker="o", markersize=3, label=opcion)
    ax.axvline(CONFIG_FINAL[variante]["refit_epochs"], color="gray", linestyle=":", linewidth=1)
    ax.set_title(f"{variante.upper()} · F1-Macro OOF por época", fontsize=10)
    ax.set_xlabel("Época")
    ax.grid(alpha=0.25)
axes[0, 0].set_ylabel("F1-Macro OOF (train)")
axes[0, 0].legend(fontsize=7)
ax = axes[0, -1]
d = CONTRASTES_TAB[CONTRASTES_TAB["variante"] == "media 3 variantes"].iloc[::-1]
ax.errorbar(d["dif_A_menos_B"], range(len(d)), xerr=[d["dif_A_menos_B"] - d["ic95_inf"], d["ic95_sup"] - d["dif_A_menos_B"]],
            fmt="o", color="k", capsize=3)
ax.axvline(0, color="gray", linewidth=0.8)
ax.set_yticks(range(len(d)), [f"{c}\n({a} − {b})" for c, a, b in zip(d["contraste"], d["A"], d["B"])], fontsize=7)
ax.set_xlabel("Diferencia de F1-Macro OOF (media 3 variantes)")
ax.set_title("Contrastes, IC 95 % bootstrap de grupos", fontsize=10)
ax.grid(axis="x", alpha=0.25)
fig.suptitle(f"{NOMBRE_EXPERIMENTO} — piloto BBFL (3 folds agrupados, solo train; línea gris = época del refit)", fontsize=11)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(PILOTO_DIR / f"{NOMBRE_EXPERIMENTO.lower()}_bbfl_resultados.{ext}", dpi=200, bbox_inches="tight")
display(fig)
plt.close(fig)

### 5.4 Decisión (regla fijada antes de ejecutar)

In [ ]:
p = CONTRASTES_TAB[CONTRASTES_TAB["contraste"] == "BBFL vs. actual (principal)"].set_index("variante")
media = p.loc["media 3 variantes"]
c1 = media["ic95_inf"] > 0
n_var = int((p.drop(index="media 3 variantes")["dif_A_menos_B"] > 0).sum())
c2 = n_var >= 2
n_folds = int(media["folds_A_mejor"].split("/")[0])
c3 = n_folds >= 6
print("Regla de decisión (fijada antes de ejecutar), BBFL frente a la configuración actual (REF):")
print(f"  (1) diferencia media 3 variantes = {media['dif_A_menos_B']:+.4f}, IC 95 % [{media['ic95_inf']:+.4f}, {media['ic95_sup']:+.4f}] "
      f"-> {'cumple' if c1 else 'no cumple'} (IC por encima de 0)")
print(f"  (2) variantes con diferencia positiva = {n_var}/3 -> {'cumple' if c2 else 'no cumple'} (al menos 2)")
print(f"  (3) folds con diferencia positiva = {media['folds_A_mejor']} -> {'cumple' if c3 else 'no cumple'} (al menos 6 de 9)")
if c1 and c2 and c3:
    print(f"\nConclusión: BBFL mejora de forma consistente a la configuración actual en {NOMBRE_EXPERIMENTO}. Merece una "
          "confirmación con Nested CV (y, si se mantiene, ampliarlo a BETO y DistilBETO). La configuración final NO se cambia "
          "con este piloto.")
else:
    print(f"\nConclusión: BBFL no cumple la regla en {NOMBRE_EXPERIMENTO}; se mantiene la configuración actual. "
          "Revisa los contrastes 2×2 para ver si el balanceo o Focal Loss por separado aportan algo.")

## 6. Cómo leer los resultados y limitaciones

- **Cifra principal:** F1-Macro OOF en la época del refit. `f1_mejor_epoca_folds` (época con mejor F1 medio de validación) es optimista, porque elige la época mirando la validación; se muestra solo como sensibilidad.
- **Contrastes:** si el IC 95 % incluye el 0, la diferencia no es concluyente. `folds_A_mejor` cuenta en cuántos folds A supera a B en el F1 de validación de ese fold.
- **Por clase:** los lotes balanceados hacen que el modelo vea un tercio irónico como si fuera la mitad, lo que suele subir el *recall* irónico y la proporción de tweets predichos como irónicos (`prop_predicha_ironica` frente a `prop_real_ironica` ≈ 0.33), a costa de la precisión irónica y del *recall* no irónico.
- **Brecha:** `brecha_f1` = F1 en el train del fold − F1 de validación; `brecha_nll` = NLL de validación − NLL del train. Con lotes balanceados los irónicos se repiten más por época, lo que puede aumentar la memorización.
- **Calibración:** Focal Loss y el balanceo cambian la escala de las probabilidades; `nll_temperatura` y `ece_temperatura` muestran cuánto se recupera con temperature scaling.

**Limitaciones del piloto.** (1) Solo 3 folds del train y la configuración final de la configuración actual: los hiperparámetros (épocas, LR, dropout) no se reoptimizaron para cada opción, lo que puede favorecer a `REF`. (2) γ = 2 y la proporción 8 + 8 se fijaron de antemano, sin ajuste. (3) El umbral es 0.5 en todas las opciones; con lotes balanceados el umbral óptimo puede ser otro, pero ajustarlo sería otra decisión que habría que validar aparte. (4) Las diferencias pequeñas pueden deberse al ruido de entrenamiento en GPU; por eso se exigen IC y consistencia entre folds y variantes. (5) Es un piloto: si BBFL cumpliera la regla, la mejora habría que confirmarla con Nested CV antes de cambiar la configuración final o usar el test.